# CS781 Phase 2 — stronger span model, Run All

Run this **in the same Kaggle session** that completed `CS781_Phase2_Span_V2_RunAll.ipynb`. Select the **two-T4 accelerator**, then **Run All**. It reads the verified Phase-1 base-03 checkpoint plus the completed v2 fold/final checkpoints and candidate caches. It creates a new `phase2_span_v3` directory; older experiments and the 0.5069 submission remain untouched. No old notebook variables are required.

The neural stage starts at each selected v2 checkpoint, unfreezes the top **12 of 24** encoder blocks, and adds a zero-initialized bilinear span scorer plus a larger boundary MLP. The initialization reproduces v2 predictions before training. Exact-span hard-negative ranking supplements the previous token and pair losses. Both T4s work on each training batch. The 24-epoch cap, early stopping, epoch checkpoints, and four-hour-45-minute guard make this a long but recoverable run. **Run All again in the same session if it pauses.**

The selection stage compares v2, v3, document-aware reranking, and a label-aware mixture on the same three held-out document folds. The mixture uses each model only where its out-of-fold label F1 supports it, and all candidates go through joint overlap decoding. The report explicitly notes selection bias. If the selected local F1 does not improve enough, no new submission is packaged.

After Run All, download `CS781_Phase2_Span_V3_Report.zip` and send it for inspection. If status is `READY_FOR_MANUAL_UPLOAD`, download `CS781_Phase2_Span_V3_Submission.zip`; also save `CS781_Phase2_Span_V3_Recovery.zip` if produced. This experiment uses only your Phase-1 weights and provided Phase-2 labeled data. AI assisted work should be disclosed according to your course rules. **The private F1, including the 0.6 target, cannot be guaranteed before evaluation.**


## Runtime and saved artifacts


In [ ]:
from __future__ import annotations

import bisect
import csv
import hashlib
import importlib.util
import json
import math
import os
import random
import shutil
import sys
import tempfile
import time
import zipfile
from collections import Counter
from pathlib import Path

import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint as activation_checkpoint

WORKING = Path('/kaggle/working')
PREVIOUS = WORKING / 'CS781_Capstone' / 'phase2_improved'
ROOT = WORKING / 'CS781_Capstone' / 'phase2_span_v3'
V1_ROOT = WORKING / 'CS781_Capstone' / 'phase2_span_v1'
V2_ROOT = WORKING / 'CS781_Capstone' / 'phase2_span_v2'
BASE_DIR = PREVIOUS / 'stages' / 'final_slot03' / 'phase1_bases' / '03'
BASE_CHECKPOINT = BASE_DIR / 'checkpoints' / 'selected_model.pt'
MODEL_SOURCE = BASE_DIR / 'model' / 'prefix_hybrid_lm.py'
BPE_SOURCE = BASE_DIR / 'model' / 'course_bpe.py'
SOURCE_DIR = PREVIOUS / 'official_phase2'
REPORT_ZIP = WORKING / 'CS781_Phase2_Span_V3_Report.zip'
SEED = 781
VERSION = 'span-top12-biaffine-rolehybrid-v3'
RUN_STARTED = time.time()
SESSION_GUARD_SECONDS = 17_100
MIN_FREE_GIB = 2.0

# This is an additional experiment. The previous six distinct methods remain
# available and are not overwritten. Only your Phase-1 base-03 weights are used.
MAX_EPOCHS = 24
MIN_EPOCHS = 12
PATIENCE = 7
VALIDATE_EVERY = 2
MIN_FINAL_CV_F1 = 0.455  # only fit the official test when the old local baseline is beaten


def file_sha(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1 << 20), b''):
            digest.update(block)
    return digest.hexdigest()


def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile('w', encoding='utf-8', dir=path.parent,
                                     suffix='.tmp', delete=False) as stream:
        temporary = Path(stream.name)
        json.dump(value, stream, ensure_ascii=False, indent=2)
        stream.write('\n')
        stream.flush()
        os.fsync(stream.fileno())
    os.replace(temporary, path)


def atomic_torch(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(dir=path.parent, suffix='.tmp', delete=False) as stream:
        temporary = Path(stream.name)
    try:
        torch.save(value, temporary)
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)


def free_gib():
    return shutil.disk_usage(WORKING).free / 2**30


def load_source(path, module_name):
    spec = importlib.util.spec_from_file_location(module_name, path)
    if spec is None or spec.loader is None:
        raise ImportError(path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module


needed = [BASE_CHECKPOINT, MODEL_SOURCE, BPE_SOURCE,
          BASE_DIR / 'tokenizer' / 'tokenizer.json',
          BASE_DIR / 'tokenizer' / 'tokenizer_config.json',
          BASE_DIR / 'tokenizer' / 'tokenizer_metadata.json',
          PREVIOUS / 'cv_folds.json', PREVIOUS / 'improvement_status.json',
          SOURCE_DIR / 'labels.txt', SOURCE_DIR / 'fold3-text.json']
for part in (1, 2):
    needed.extend([SOURCE_DIR / f'fold{part}-text.json',
                   SOURCE_DIR / f'fold{part}-gold.json'])
missing = [str(path) for path in needed if not path.is_file()]
if missing:
    raise FileNotFoundError('The saved Kaggle session is incomplete:\n' + '\n'.join(missing))
if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
    raise RuntimeError('Select the Kaggle two-T4 accelerator before running this notebook.')
if free_gib() < MIN_FREE_GIB:
    raise RuntimeError(f'Only {free_gib():.2f} GiB free; stop before writing checkpoints.')
ROOT.mkdir(parents=True, exist_ok=True)
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
try:
    torch.set_float32_matmul_precision('high')
except Exception:
    pass
print('Saved run:', PREVIOUS)
print('New run:', ROOT)
print('Disk free:', f'{free_gib():.2f} GiB')
for device_id in (0, 1):
    available, total = torch.cuda.mem_get_info(device_id)
    print(f'cuda:{device_id}: {torch.cuda.get_device_name(device_id)} '
          f'{available/2**30:.2f}/{total/2**30:.2f} GiB free')
print('FP32 training; both GPUs via DataParallel. Old artifacts are read only.')


try:
    from sklearn.ensemble import HistGradientBoostingClassifier
except ImportError as exc:
    raise RuntimeError('This Kaggle session requires standard scikit-learn') from exc


## Official data and Phase-1 base


In [ ]:
fold_manifest = read_json(PREVIOUS / 'cv_folds.json')
old_status = read_json(PREVIOUS / 'improvement_status.json')
OLD_OOF = float(old_status['completed']['3']['cv']['baseline_oof_macro_f1'])
LABELS = tuple(line.strip() for line in (SOURCE_DIR / 'labels.txt').read_text(encoding='utf-8').splitlines()
               if line.strip())
if len(LABELS) != 12 or len(set(LABELS)) != 12:
    raise RuntimeError('The 12 official labels were not found')
LABEL_ID = {name: i for i, name in enumerate(LABELS)}
texts, gold = {}, {}
for part in (1, 2):
    part_text = read_json(SOURCE_DIR / f'fold{part}-text.json')
    part_gold = read_json(SOURCE_DIR / f'fold{part}-gold.json')
    if set(part_text) != set(part_gold) or set(texts) & set(part_text):
        raise RuntimeError('Official labeled document IDs are inconsistent')
    texts.update(part_text)
    gold.update({doc_id: [(int(row[0]), int(row[1]), row[2]) for row in rows]
                 for doc_id, rows in part_gold.items()})
test_texts = read_json(SOURCE_DIR / 'fold3-text.json')
FOLDS = [tuple(str(doc_id) for doc_id in group) for group in fold_manifest['folds']]
if len(FOLDS) != 3 or set(x for group in FOLDS for x in group) != set(texts) or sum(map(len, FOLDS)) != len(texts):
    raise RuntimeError('The saved document folds do not partition the official training data')
if set(test_texts) & set(texts):
    raise RuntimeError('Private test document IDs overlap public labeled training IDs')
for doc_id, rows in gold.items():
    text = texts[doc_id]
    for a, b, name in rows:
        if name not in LABEL_ID or not 0 <= a < b <= len(text):
            raise RuntimeError(f'Invalid gold span in {doc_id}: {(a,b,name)}')

model_sha = file_sha(MODEL_SOURCE)
bpe_sha = file_sha(BPE_SOURCE)
base_sha = file_sha(BASE_CHECKPOINT)
model_module = load_source(MODEL_SOURCE, 'cs781_span_base_' + model_sha[:12])
bpe_module = load_source(BPE_SOURCE, 'cs781_span_bpe_' + bpe_sha[:12])
tokenizer = bpe_module.CourseBPETokenizer.from_pretrained(BASE_DIR / 'tokenizer')
tokenizer_meta = read_json(BASE_DIR / 'tokenizer' / 'tokenizer_metadata.json')
base_record = torch.load(BASE_CHECKPOINT, map_location='cpu', weights_only=True)
if base_record.get('model_source_sha256') != model_sha:
    raise RuntimeError('Base-03 checkpoint/model source SHA256 mismatch')
if base_record.get('tokenizer_fingerprint') != tokenizer_meta.get('fingerprint'):
    raise RuntimeError('Base-03 checkpoint/tokenizer fingerprint mismatch')
if len(tokenizer) != int(base_record['model_config']['vocab_size']):
    raise RuntimeError('Base-03 tokenizer vocabulary mismatch')
BASE_CONFIG = base_record['model_config']
del base_record

data_hash = hashlib.sha256(json.dumps({
    'folds': fold_manifest, 'labels': LABELS,
    'sources': {path.name: file_sha(path) for path in needed if path.parent == SOURCE_DIR}
}, sort_keys=True).encode()).hexdigest()

CONFIG = {
    'version': VERSION, 'base_id': '03', 'adapter': 'LoRA query/value',
    'window_tokens': 384, 'window_stride': 288,
    'max_span_tokens': 64, 'lora_rank': 8, 'lora_alpha': 16.0,
    'batch_size': 16, 'epochs': MAX_EPOCHS, 'min_epochs': MIN_EPOCHS,
    'patience': PATIENCE, 'validate_every': VALIDATE_EVERY,
    'lora_lr': 2e-5, 'head_lr': 5e-5, 'weight_decay': 0.01,
    'warmup_fraction': 0.05, 'gradient_clip_norm': 1.0,
    'head_dropout': 0.15, 'pair_hidden': 128, 'refine_hidden': 384, 'unfreeze_top_blocks': 4,
    'v3_unfreeze_top_blocks': 12, 'v3_residual_hidden': 768,
    'start_topk': 64, 'end_topk': 64, 'pair_topk_per_label': 384,
    'keep_per_label_per_window': 48,
    'inference_window_batch_size': 8,
    'default_threshold': 0.45,
    'min_final_cv_f1': MIN_FINAL_CV_F1,
    'warm_start': 'verified phase2_span_v2 fold and final adapters',
    'seed': SEED, 'precision': 'float32', 'activation_checkpointing': True,
    'base_checkpoint_sha256': base_sha, 'model_source_sha256': model_sha,
    'bpe_source_sha256': bpe_sha, 'tokenizer_fingerprint': tokenizer_meta['fingerprint'],
    'data_hash': data_hash,
}
CONFIG_HASH = hashlib.sha256(json.dumps(CONFIG, sort_keys=True).encode()).hexdigest()
config_path = ROOT / 'experiment_config.json'
if config_path.exists() and read_json(config_path) != CONFIG:
    raise RuntimeError('Existing span-run configuration differs; refusing unsafe resume')
atomic_json(config_path, CONFIG)
print('Documents:', len(texts), 'test:', len(test_texts), 'folds:', [len(group) for group in FOLDS])
print('Old slot-03 default OOF F1:', f'{OLD_OOF:.6f}')
print('New experiment SHA:', CONFIG_HASH)



## Token alignment and windows


In [ ]:
ENCODINGS = {}
ALIGNED = {}
alignment_counts = Counter()


def encode_doc(doc_id, text):
    if doc_id not in ENCODINGS:
        encoded = tokenizer.backend.encode(text, add_special_tokens=False)
        ids = tuple(int(x) for x in encoded.ids)
        offsets = tuple((int(a), int(b)) for a, b in encoded.offsets)
        if (not ids and text) or len(ids) != len(offsets) or any(a >= b for a, b in offsets):
            raise RuntimeError(f'Invalid token offsets in document {doc_id}')
        ENCODINGS[doc_id] = (ids, offsets)
    return ENCODINGS[doc_id]


def align_gold(doc_id):
    if doc_id in ALIGNED:
        return ALIGNED[doc_id]
    ids, offsets = encode_doc(doc_id, texts[doc_id])
    starts = [a for a, _ in offsets]
    ends = [b for _, b in offsets]
    aligned = []
    for a, b, name in gold[doc_id]:
        i = bisect.bisect_right(ends, a)
        j = bisect.bisect_left(starts, b) - 1
        if not (0 <= i <= j < len(ids) and starts[i] <= a and b <= ends[j]):
            raise RuntimeError(f'Gold span has no token alignment in {doc_id}: {(a,b,name)}')
        if j - i + 1 > CONFIG['max_span_tokens']:
            raise RuntimeError(f'Gold span exceeds max token width in {doc_id}: {(a,b,name)}')
        left, right = starts[i], ends[j]
        while left < right and texts[doc_id][left].isspace():
            left += 1
        while right > left and texts[doc_id][right - 1].isspace():
            right -= 1
        alignment_counts['total'] += 1
        alignment_counts['trim_exact'] += int((left, right) == (a, b))
        aligned.append((i, j, LABEL_ID[name], a, b))
    ALIGNED[doc_id] = tuple(aligned)
    return ALIGNED[doc_id]


def window_bounds(length, size, stride):
    if length <= size:
        return [(0, length)]
    starts = list(range(0, length - size + 1, stride))
    if starts[-1] != length - size:
        starts.append(length - size)
    return [(start, min(start + size, length)) for start in starts]


def make_windows(doc_ids, training):
    rows, rare_rows, uncovered = [], [], []
    rare = {LABEL_ID[name] for name in ('APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT')}
    for doc_id in sorted(doc_ids):
        ids, offsets = encode_doc(doc_id, texts[doc_id] if doc_id in texts else test_texts[doc_id])
        spans = align_gold(doc_id) if training else ()
        bounds = window_bounds(len(ids), CONFIG['window_tokens'], CONFIG['window_stride'])
        covered = [False] * len(spans)
        seen_bounds = set()
        def add_window(lo, hi):
            if (lo, hi) in seen_bounds:
                return
            seen_bounds.add((lo, hi))
            targets = []
            for position, (i, j, label, a, b) in enumerate(spans):
                if lo <= i and j < hi:
                    targets.append((i - lo, j - lo, label, a, b))
                    covered[position] = True
            row = {'id': f'{doc_id}__{lo:06d}', 'doc_id': doc_id, 'token_lo': lo,
                   'ids': ids[lo:hi], 'offsets': offsets[lo:hi], 'targets': tuple(targets)}
            rows.append(row)
            if any(t[2] in rare for t in targets):
                rare_rows.append(row)
        for lo, hi in bounds:
            add_window(lo, hi)
        if training:
            for position, (i, j, label, a, b) in enumerate(spans):
                if covered[position]:
                    continue
                lo = max(0, min(i, len(ids) - CONFIG['window_tokens']))
                add_window(lo, min(lo + CONFIG['window_tokens'], len(ids)))
            for position, flag in enumerate(covered):
                if not flag:
                    uncovered.append((doc_id, spans[position]))
    if uncovered:
        raise RuntimeError('Gold spans missing from training windows: ' + repr(uncovered[:5]))
    if training and rare_rows:
        # One extra visit for at most a quarter of the original windows.
        sample_count = min(len(rare_rows), len(rows) // 4)
        rng = random.Random(SEED + len(rows))
        for index, row in enumerate(rng.sample(rare_rows, sample_count)):
            rows.append(dict(row, id=row['id'] + f'__rare{index:04d}'))
    return rows


for doc_id in texts:
    align_gold(doc_id)
if alignment_counts['total'] != sum(map(len, gold.values())):
    raise RuntimeError('Gold alignment audit is incomplete')
print('Gold token alignment:', alignment_counts['total'], 'of', alignment_counts['total'])
print('Whitespace-trim token boundary exact rate:',
      f"{alignment_counts['trim_exact']/alignment_counts['total']:.4f}")
print('Longest public document tokens:', max(len(encode_doc(doc_id, text)[0]) for doc_id, text in texts.items()))



## Span model and loss


In [ ]:
class AdditiveLoRA(nn.Module):
    def __init__(self, original, rank, alpha):
        super().__init__()
        self.base = original
        self.a = nn.Linear(original.in_features, rank, bias=False)
        self.b = nn.Linear(rank, original.out_features, bias=False)
        nn.init.kaiming_uniform_(self.a.weight, a=math.sqrt(5))
        nn.init.zeros_(self.b.weight)
        self.scale = alpha / rank

    def forward(self, x):
        return self.base(x) + self.b(self.a(x)) * self.scale


class SpanExtractor(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base = base
        for parameter in base.parameters():
            parameter.requires_grad_(False)
        changed = 0
        for block in base.blocks:
            for name in ('query', 'value'):
                layer = getattr(block.attention, name)
                if not isinstance(layer, nn.Linear):
                    raise RuntimeError(f'Unexpected base-03 attention projection {name}')
                setattr(block.attention, name,
                        AdditiveLoRA(layer, CONFIG['lora_rank'], CONFIG['lora_alpha']))
                changed += 1
        if changed != 2 * len(base.blocks):
            raise RuntimeError('LoRA was not applied to every base-03 attention block')
        hidden = base.hidden_size
        self.dropout = nn.Dropout(CONFIG['head_dropout'])
        self.start_head = nn.Linear(hidden, len(LABELS))
        self.end_head = nn.Linear(hidden, len(LABELS))
        self.width_embedding = nn.Embedding(CONFIG['max_span_tokens'] + 1, 16)
        self.pair_head = nn.Sequential(
            nn.Linear(3 * hidden + 16, CONFIG['pair_hidden']),
            nn.GELU(), nn.Dropout(CONFIG['head_dropout']),
            nn.Linear(CONFIG['pair_hidden'], len(LABELS)))

    def features(self, input_ids, attention_mask):
        base = self.base
        length = input_ids.shape[1]
        visibility = base._prefix_hybrid_visibility(attention_mask, attention_mask.sum(1))
        states = base.embedding_dropout(base.token_embedding(input_ids) * math.sqrt(base.hidden_size))
        cosine, sine = base.rotary(length, states.dtype)
        for block in base.blocks:
            if self.training and torch.is_grad_enabled() and CONFIG['activation_checkpointing']:
                states = activation_checkpoint(block, states, visibility, cosine, sine,
                                               use_reentrant=False)
            else:
                states = block(states, visibility, cosine, sine)
        return self.dropout(base.final_norm(states))

    def score_pairs(self, hidden, pair_index):
        width = hidden.shape[-1]
        left = pair_index[..., 0].unsqueeze(-1).expand(-1, -1, width)
        right = pair_index[..., 1].unsqueeze(-1).expand(-1, -1, width)
        start = hidden.gather(1, left)
        end = hidden.gather(1, right)
        distance = (pair_index[..., 1] - pair_index[..., 0]).clamp(
            min=0, max=CONFIG['max_span_tokens'])
        features = torch.cat((start, end, start * end,
                              self.width_embedding(distance)), dim=-1)
        return self.pair_head(features)

    def forward(self, input_ids, attention_mask, pair_index=None, return_hidden=False):
        hidden = self.features(input_ids, attention_mask)
        output = {'start': self.start_head(hidden), 'end': self.end_head(hidden)}
        if pair_index is not None:
            output['pair'] = self.score_pairs(hidden, pair_index)
        if return_hidden:
            output['hidden'] = hidden
        return output


def fresh_model():
    record = torch.load(BASE_CHECKPOINT, map_location='cpu', weights_only=True)
    if record.get('model_source_sha256') != model_sha or record.get('tokenizer_fingerprint') != tokenizer_meta['fingerprint']:
        raise RuntimeError('Phase-1 base changed while this notebook was running')
    base = model_module.build_model(record['model_config'])
    base.load_state_dict(record['model_state'], strict=True)
    del record
    model = SpanExtractor(base)
    adapted = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'LoRA + span-head trainable parameters: {adapted:,} / '
          f'{sum(p.numel() for p in model.parameters()):,}')
    return model


def trainable_state(model):
    names = {name for name, parameter in model.named_parameters() if parameter.requires_grad}
    return {name: value.detach().cpu().clone() for name, value in model.state_dict().items()
            if name in names}


def load_trainable_state(model, saved):
    names = {name for name, parameter in model.named_parameters() if parameter.requires_grad}
    if set(saved) != names:
        raise RuntimeError('Adapter/head state keys differ from this model source')
    current = model.state_dict()
    for name, value in saved.items():
        if current[name].shape != value.shape:
            raise RuntimeError(f'Adapter/head shape mismatch: {name}')
        current[name] = value
    model.load_state_dict(current, strict=True)


def label_weights(train_ids):
    counts = Counter(name for doc_id in train_ids for _, _, name in gold[doc_id])
    positive = sorted(counts.values())
    median = positive[len(positive)//2]
    return torch.tensor([min(3.0, max(0.6, math.sqrt(median / max(1, counts[name]))))
                         for name in LABELS], dtype=torch.float32)


def make_pair_examples(row, epoch):
    length = len(row['ids'])
    positives = {(int(i), int(j), int(label)) for i, j, label, a, b in row['targets']}
    pairs = {(i, j) for i, j, _ in positives}
    for i, j, _ in positives:
        for di in (-2, -1, 0, 1, 2):
            for dj in (-2, -1, 0, 1, 2):
                x, y = i + di, j + dj
                if 0 <= x <= y < length and y - x < CONFIG['max_span_tokens']:
                    pairs.add((x, y))
    stable = int(hashlib.sha256(row['id'].encode()).hexdigest()[:8], 16)
    rng = random.Random(SEED + stable + epoch * 100_003)
    for _ in range(max(50, 3 * len(positives))):
        x = rng.randrange(length)
        y = min(length - 1, x + rng.randrange(CONFIG['max_span_tokens']))
        pairs.add((x, y))
    if len(pairs) > 320:
        # Keep every positive and close boundary perturbation first. Random
        # negatives are then capped to keep DataParallel batches predictable.
        priority = sorted(pairs, key=lambda pair: min(
            (abs(pair[0]-i) + abs(pair[1]-j) for i, j, _ in positives),
            default=10_000))
        pairs = set(priority[:320])
    ordered = sorted(pairs)
    targets = [[int((i, j, label) in positives) for label in range(len(LABELS))]
               for i, j in ordered]
    return ordered, targets


def collate_windows(rows, epoch):
    batch_size = len(rows)
    length = max(len(row['ids']) for row in rows)
    input_ids = torch.full((batch_size, length), tokenizer.pad_token_id, dtype=torch.long)
    attention_mask = torch.zeros((batch_size, length), dtype=torch.bool)
    starts = torch.zeros((batch_size, length, len(LABELS)), dtype=torch.float32)
    ends = torch.zeros_like(starts)
    examples = [make_pair_examples(row, epoch) for row in rows]
    max_pairs = max(len(item[0]) for item in examples)
    pair_index = torch.zeros((batch_size, max_pairs, 2), dtype=torch.long)
    pair_targets = torch.zeros((batch_size, max_pairs, len(LABELS)), dtype=torch.float32)
    pair_mask = torch.zeros((batch_size, max_pairs), dtype=torch.bool)
    for batch_index, row in enumerate(rows):
        size = len(row['ids'])
        input_ids[batch_index, :size] = torch.tensor(row['ids'], dtype=torch.long)
        attention_mask[batch_index, :size] = True
        for i, j, label, a, b in row['targets']:
            starts[batch_index, i, label] = 1
            ends[batch_index, j, label] = 1
        pairs, targets = examples[batch_index]
        pair_index[batch_index, :len(pairs)] = torch.tensor(pairs, dtype=torch.long)
        pair_targets[batch_index, :len(pairs)] = torch.tensor(targets, dtype=torch.float32)
        pair_mask[batch_index, :len(pairs)] = True
    return {'input_ids': input_ids, 'attention_mask': attention_mask,
            'start_targets': starts, 'end_targets': ends,
            'pair_index': pair_index, 'pair_targets': pair_targets,
            'pair_mask': pair_mask}


def balanced_loss(logits, targets, valid_mask, class_weights):
    valid = valid_mask.unsqueeze(-1).expand_as(targets)
    positive = valid & targets.bool()
    negative = valid & ~targets.bool()
    weight = class_weights.view(*([1] * (logits.ndim - 1)), -1)
    positive_loss = (F.softplus(-logits.float()) * positive * weight).sum() / (
        (positive * weight).sum().clamp_min(1))
    negative_values = F.softplus(logits.float())[negative]
    if negative_values.numel():
        k = min(negative_values.numel(), max(64, int(positive.sum().item()) * 5))
        negative_loss = negative_values.topk(k).values.mean()
    else:
        negative_loss = logits.new_zeros(())
    return positive_loss + 0.35 * negative_loss


def task_loss(output, batch, class_weights):
    start_loss = balanced_loss(output['start'], batch['start_targets'],
                               batch['attention_mask'], class_weights)
    end_loss = balanced_loss(output['end'], batch['end_targets'],
                             batch['attention_mask'], class_weights)
    pair_loss = balanced_loss(output['pair'], batch['pair_targets'],
                              batch['pair_mask'], class_weights)
    return (start_loss + end_loss + pair_loss) / 3



## Two-GPU inference and exact F1


In [ ]:
def inference_batch(rows):
    width = max(len(row['ids']) for row in rows)
    ids = torch.full((len(rows), width), tokenizer.pad_token_id, dtype=torch.long)
    masks = torch.zeros((len(rows), width), dtype=torch.bool)
    for index, row in enumerate(rows):
        ids[index, :len(row['ids'])] = torch.tensor(row['ids'], dtype=torch.long)
        masks[index, :len(row['ids'])] = True
    return ids.to('cuda:0', non_blocking=True), masks.to('cuda:0', non_blocking=True)


def candidate_pairs(start_logits, end_logits, length):
    """Prune the cross product before the more expensive learned pair head."""
    by_label = {}
    unique_pairs = set()
    start_logits, end_logits = start_logits[:length].float(), end_logits[:length].float()
    for label in range(len(LABELS)):
        starts = torch.topk(start_logits[:, label], min(CONFIG['start_topk'], length)).indices
        ends = torch.topk(end_logits[:, label], min(CONFIG['end_topk'], length)).indices
        span_widths = ends[None, :] - starts[:, None]
        allowed = (span_widths >= 0) & (span_widths < CONFIG['max_span_tokens'])
        pair_count = min(CONFIG['pair_topk_per_label'], int(allowed.sum()))
        if pair_count:
            scores = start_logits[starts, label][:, None] + end_logits[ends, label][None, :]
            scores = scores.masked_fill(~allowed, float('-inf'))
            values, indexes = torch.topk(scores.flatten(), pair_count)
            start_positions = starts[indexes // len(ends)]
            end_positions = ends[indexes % len(ends)]
            kept = [(float(v), int(i), int(j))
                    for v, i, j in zip(values, start_positions, end_positions)]
        else:
            kept = []
        by_label[label] = kept
        unique_pairs.update((i, j) for _, i, j in kept)
    return by_label, sorted(unique_pairs)


def score_window_candidates(model, hidden, start_logits, end_logits, row, source_text):
    length = len(row['ids'])
    by_label, pairs = candidate_pairs(start_logits, end_logits, length)
    if not pairs:
        return []
    pair_scores = []
    for begin in range(0, len(pairs), 1024):
        group = pairs[begin:begin + 1024]
        indexes = torch.tensor(group, dtype=torch.long, device='cuda:0').unsqueeze(0)
        pair_scores.append(model.score_pairs(hidden.unsqueeze(0), indexes)[0].float().cpu())
    pair_scores = torch.cat(pair_scores, dim=0)
    pair_positions = {pair: index for index, pair in enumerate(pairs)}
    scored = []
    offsets = row['offsets']
    for label, proposals in by_label.items():
        ranked = []
        for _, i, j in proposals:
            pair_logit = float(pair_scores[pair_positions[(i, j)], label])
            combined_logit = (float(start_logits[i, label])
                              + float(end_logits[j, label]) + pair_logit) / 3
            confidence = 1 / (1 + math.exp(-max(-50, min(50, combined_logit))))
            ranked.append((confidence, i, j))
        ranked.sort(reverse=True)
        for confidence, i, j in ranked[:CONFIG['keep_per_label_per_window']]:
            a, b = offsets[i][0], offsets[j][1]
            while a < b and source_text[a].isspace():
                a += 1
            while b > a and source_text[b - 1].isspace():
                b -= 1
            if 0 <= a < b <= len(source_text):
                scored.append((a, b, LABELS[label], confidence))
    return scored


def predict_documents(model, parallel, document_ids, document_texts,
                      existing=None, on_document=None):
    was_training = model.training
    model.eval()
    predictions = dict(existing or {})
    try:
        with torch.inference_mode():
            for doc_id in sorted(document_ids):
                if doc_id in predictions:
                    continue
                text = document_texts[doc_id]
                ids, offsets = encode_doc(doc_id, text)
                if not ids:
                    predictions[doc_id] = []
                    if on_document is not None:
                        on_document(predictions)
                    continue
                bounds = window_bounds(len(ids), CONFIG['window_tokens'], CONFIG['window_stride'])
                rows = [{'ids': ids[lo:hi], 'offsets': offsets[lo:hi]}
                        for lo, hi in bounds]
                found = {}
                batch_size = CONFIG['inference_window_batch_size']
                cursor = 0
                while cursor < len(rows):
                    group = rows[cursor:cursor + batch_size]
                    input_ids, masks = inference_batch(group)
                    try:
                        # DataParallel cannot scatter a one-row batch across two GPUs.
                        outputs = (parallel(input_ids, masks, return_hidden=True)
                                   if len(group) > 1 else
                                   model(input_ids, masks, return_hidden=True))
                    except RuntimeError as exc:
                        if 'out of memory' not in str(exc).lower() or batch_size == 1:
                            raise
                        del input_ids, masks
                        torch.cuda.empty_cache()
                        batch_size = max(1, batch_size // 2)
                        continue
                    for row_number, row in enumerate(group):
                        candidates = score_window_candidates(
                            model, outputs['hidden'][row_number],
                            outputs['start'][row_number].detach().cpu(),
                            outputs['end'][row_number].detach().cpu(), row, text)
                        for a, b, name, confidence in candidates:
                            key = (a, b, name)
                            found[key] = max(found.get(key, 0.0), confidence)
                    cursor += len(group)
                predictions[doc_id] = [[a, b, name, float(score)]
                                       for (a, b, name), score in sorted(
                                           found.items(), key=lambda item: -item[1])]
                if on_document is not None:
                    on_document(predictions)
    finally:
        if was_training:
            model.train()
    return predictions


def decode(candidate_map, thresholds=None):
    thresholds = thresholds or {}
    results = {}
    for doc_id, candidates in candidate_map.items():
        accepted, starts = [], []
        for a, b, name, score in candidates:
            if float(score) < float(thresholds.get(name, CONFIG['default_threshold'])):
                continue
            index = bisect.bisect_left(starts, a)
            if ((index and accepted[index - 1][1] > a)
                    or (index < len(accepted) and accepted[index][0] < b)):
                continue
            starts.insert(index, int(a))
            accepted.insert(index, (int(a), int(b), name))
        results[doc_id] = [list(item) for item in accepted]
    return results


def strict_macro(candidate_map, reference_ids, thresholds=None):
    if set(candidate_map) != set(reference_ids):
        raise RuntimeError('Candidate IDs differ from evaluation document IDs')
    decoded = decode(candidate_map, thresholds)
    per_label = {}
    for name in LABELS:
        predicted = {(doc_id, a, b) for doc_id, rows in decoded.items()
                     for a, b, label in rows if label == name}
        expected = {(doc_id, a, b) for doc_id in reference_ids
                    for a, b, label in gold[doc_id] if label == name}
        raw = {(doc_id, int(a), int(b)) for doc_id, rows in candidate_map.items()
               for a, b, label, score in rows if label == name}
        tp = len(predicted & expected)
        p = tp / len(predicted) if predicted else 0.0
        r = tp / len(expected) if expected else 0.0
        f1 = 2 * p * r / (p + r) if p + r else 0.0
        hits = len(raw & expected)
        per_label[name] = {'gold': len(expected), 'predicted': len(predicted),
                           'tp': tp, 'precision': p, 'recall': r, 'f1': f1,
                           'exact_candidate': hits,
                           'exact_candidate_recall': hits / len(expected) if expected else 0.0}
    return {'macro_f1': sum(row['f1'] for row in per_label.values()) / len(LABELS),
            'per_label': per_label}



## V2 provenance and larger v3 model


In [ ]:
class EnhancedSpanExtractor(SpanExtractor):
    def __init__(self, base):
        super().__init__(base)
        width = base.hidden_size
        self.pair_refine = nn.Sequential(
            nn.Linear(5 * width + 16, CONFIG['refine_hidden']),
            nn.GELU(), nn.Dropout(0.18),
            nn.Linear(CONFIG['refine_hidden'], len(LABELS)))
        nn.init.zeros_(self.pair_refine[-1].weight)
        nn.init.zeros_(self.pair_refine[-1].bias)
        for block in base.blocks[-CONFIG['unfreeze_top_blocks']:]:
            for parameter in block.parameters():
                parameter.requires_grad_(True)

    def score_pairs(self, hidden, pair_index):
        width = hidden.shape[-1]
        start_index = pair_index[..., 0]
        end_index = pair_index[..., 1]
        def take(index):
            return hidden.gather(1, index.unsqueeze(-1).expand(-1, -1, width))
        start, end = take(start_index), take(end_index)
        left = take((start_index - 1).clamp_min(0))
        right = take((end_index + 1).clamp_max(hidden.shape[1] - 1))
        distance = (end_index - start_index).clamp(min=0, max=CONFIG['max_span_tokens'])
        width_feature = self.width_embedding(distance)
        original = torch.cat((start, end, start * end, width_feature), dim=-1)
        boundary = torch.cat((start, end, left, right, start * end, width_feature), dim=-1)
        return self.pair_head(original) + self.pair_refine(boundary)

def combine_candidate_scores(v1_rows, v2_rows):
    merged = {}
    for a, b, label, score in v1_rows:
        merged[(int(a), int(b), label)] = [float(score), 0.0]
    for a, b, label, score in v2_rows:
        key = (int(a), int(b), label)
        merged.setdefault(key, [0.0, 0.0])[1] = float(score)
    return merged

def document_features(doc_id, text, merged):
    n = max(1, len(text))
    strong_frequency = Counter()
    early_label = {}
    start_max, end_max, exact_labels = {}, {}, {}
    for (a, b, label), (s1, s2) in merged.items():
        peak = max(s1, s2)
        key = ' '.join(text[a:b].casefold().split())
        if peak >= 0.35 and key:
            strong_frequency[key] += 1
        if a / n < 0.25 and label in ('APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL'):
            early_label[(key, label)] = max(early_label.get((key, label), 0.0), peak)
        start_max[a] = max(start_max.get(a, 0.0), peak)
        end_max[b] = max(end_max.get(b, 0.0), peak)
        exact_labels.setdefault((a, b), {})[label] = peak
    rows, features = [], []
    for (a, b, label), (s1, s2) in merged.items():
        phrase = text[a:b]
        normalized = ' '.join(phrase.casefold().split())
        before = text[max(0, a-90):a].casefold()
        after = text[b:min(len(text), b+60)].casefold()
        nearby = before + ' ' + after
        position = a / n
        length = b - a
        peak = max(s1, s2)
        other = max((score for lab, score in exact_labels[(a, b)].items()
                     if lab != label), default=0.0)
        role = label in ('APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL')
        opposite = {'APP': 'RESP', 'RESP': 'APP',
                    'A.COUNSEL': 'R.COUNSEL', 'R.COUNSEL': 'A.COUNSEL'}.get(label)
        values = [
            s1, s2, peak, min(s1, s2), abs(s1-s2),
            math.log(max(1e-5, s1) / max(1e-5, 1-s1)),
            math.log(max(1e-5, s2) / max(1e-5, 1-s2)),
            position, float(position < .1), float(position < .25),
            math.log1p(length), min(length, 150) / 150,
            min(phrase.count(' ') + 1, 20) / 20,
            float(phrase[:1].isupper()),
            sum(c.isdigit() for c in phrase) / max(1, length),
            float(any(c in phrase for c in '.,()[]:-/')),
            math.log1p(min(strong_frequency[normalized], 100)),
            early_label.get((normalized, label), 0.0) if role else 0.0,
            early_label.get((normalized, opposite), 0.0) if opposite else 0.0,
            start_max[a] - peak, end_max[b] - peak, other,
            float('appellant' in nearby or 'petitioner' in nearby),
            float('respondent' in nearby or 'defendant' in nearby),
            float('advocate' in nearby or 'counsel' in nearby),
            float('witness' in nearby or 'deposed' in nearby),
            float('judge' in nearby or 'justice' in nearby),
            float(' v. ' in nearby or ' vs ' in nearby or 'versus' in nearby),
            float('section' in nearby or 'article' in nearby),
            float('court' in nearby),
            float('reported' in nearby or 'citation' in nearby),
            float(phrase.isupper()),
        ]
        values.extend(float(label == name) for name in LABELS)
        rows.append((a, b, label, s1, s2))
        features.append(values)
    if not features:
        return rows, np.empty((0, 32 + len(LABELS)), dtype=np.float32)
    return rows, np.asarray(features, dtype=np.float32)

def build_feature_table(doc_ids, v1_map, v2_map, document_texts, with_gold):
    features, rows_by_doc, ranges, targets = [], {}, {}, []
    cursor = 0
    for doc_id in sorted(doc_ids):
        merged = combine_candidate_scores(v1_map[doc_id], v2_map[doc_id])
        rows, matrix = document_features(doc_id, document_texts[doc_id], merged)
        rows_by_doc[doc_id] = rows
        features.append(matrix)
        ranges[doc_id] = (cursor, cursor + len(rows))
        cursor += len(rows)
        if with_gold:
            exact = {(int(row[0]), int(row[1]), row[2]) for row in gold[doc_id]}
            targets.extend(int((a, b, label) in exact) for a, b, label, _, _ in rows)
    if not features or any(matrix.shape[1] != features[0].shape[1] for matrix in features):
        raise RuntimeError('Candidate feature width changed between documents')
    X = np.concatenate(features, axis=0)
    y = np.asarray(targets, dtype=np.uint8) if with_gold else None
    if not np.isfinite(X).all() or (with_gold and len(y) != len(X)):
        raise RuntimeError('Document candidate features are invalid')
    return X, y, rows_by_doc, ranges

def train_reranker(X, y, ranges, training_ids):
    indices = []
    rng = np.random.default_rng(SEED + len(training_ids))
    for doc_id in training_ids:
        lo, hi = ranges[doc_id]
        if lo == hi:
            continue
        selection = np.arange(lo, hi)
        score = np.maximum(X[lo:hi, 0], X[lo:hi, 1])
        keep = (y[lo:hi] == 1) | (score >= 0.22) | (rng.random(hi-lo) < 0.045)
        indices.append(selection[keep])
    indices = np.concatenate(indices)
    if len(indices) == 0 or y[indices].sum() < 100:
        raise RuntimeError('Reranker has too few positive training candidates')
    weights = np.where(y[indices] == 1, 5.0, 1.0).astype(np.float32)
    model = HistGradientBoostingClassifier(
        learning_rate=0.075, max_iter=120, max_leaf_nodes=31,
        min_samples_leaf=35, l2_regularization=2.0,
        early_stopping=False, random_state=SEED)
    model.fit(X[indices], y[indices], sample_weight=weights)
    print('Reranker training:', len(indices), 'candidates,',
          int(y[indices].sum()), 'positive exact spans', flush=True)
    return model

def rerank_documents(model, X, rows_by_doc, ranges, doc_ids):
    output = {}
    for doc_id in doc_ids:
        lo, hi = ranges[doc_id]
        if lo == hi:
            output[doc_id] = []
            continue
        probabilities = model.predict_proba(X[lo:hi])[:, 1]
        rows = [[a, b, label, float(probabilities[i])]
                for i, (a, b, label, _, _) in enumerate(rows_by_doc[doc_id])
                if probabilities[i] >= .025]
        rows.sort(key=lambda row: -row[3])
        output[doc_id] = rows
    return output

def fast_macro(candidate_map, thresholds):
    predicted = decode(candidate_map, thresholds)
    values = []
    for name in LABELS:
        p = {(doc_id, a, b) for doc_id, rows in predicted.items()
             for a, b, label in rows if label == name}
        g = {(doc_id, a, b) for doc_id in candidate_map
             for a, b, label in gold[doc_id] if label == name}
        tp = len(p & g)
        values.append(2 * tp / (len(p) + len(g)) if p or g else 0.0)
    return sum(values) / len(LABELS)

def calibrate(candidate_map, starting_threshold=.45):
    thresholds = {label: starting_threshold for label in LABELS}
    score = fast_macro(candidate_map, thresholds)
    grid = (.05, .10, .15, .20, .30, .40, .50, .60, .70, .80)
    for label in LABELS:
        best_t, best_score = thresholds[label], score
        for trial in grid:
            proposal = dict(thresholds, **{label: trial})
            trial_score = fast_macro(candidate_map, proposal)
            if trial_score > best_score + 1e-8:
                best_t, best_score = trial, trial_score
        thresholds[label], score = best_t, best_score
    return thresholds, score

import pickle
import numpy as np

V2_CONFIG = read_json(V2_ROOT / 'experiment_config.json')
V2_HASH = hashlib.sha256(json.dumps(V2_CONFIG, sort_keys=True).encode()).hexdigest()
for key, current in [
    ('base_checkpoint_sha256', base_sha), ('model_source_sha256', model_sha),
    ('bpe_source_sha256', bpe_sha),
    ('tokenizer_fingerprint', tokenizer_meta['fingerprint']), ('data_hash', data_hash),
]:
    if V2_CONFIG.get(key) != current:
        raise RuntimeError(f'The v2 experiment has a different {key}')
V2_REPORT = read_json(V2_ROOT / 'report.json')
V2_COMPARISON = read_json(V2_ROOT / 'model_comparison.json')
V2_FINAL = read_json(V2_ROOT / 'final' / 'summary.json')
V2_TEST_PATH = V2_ROOT / 'final' / 'test_candidates.partial.json'
if (V2_REPORT.get('status') != 'READY_FOR_MANUAL_UPLOAD' or
        V2_FINAL.get('status') != 'complete' or not V2_TEST_PATH.is_file()):
    raise RuntimeError('Complete the v2 Run All notebook in this same Kaggle session first')
V2_BASELINE = float(V2_COMPARISON['reranker_calibrated_oof'])
if abs(V2_BASELINE - 0.45297112108361587) > 1e-7:
    raise RuntimeError('Unexpected v2 out-of-fold baseline; inspect artifacts before continuing')


def selected_v2(fold_index):
    folder = V2_ROOT / f'fold{fold_index}'
    best = read_json(folder / 'best.json')
    if best.get('config_hash') != V2_HASH or best.get('fold') != fold_index:
        raise RuntimeError(f'Fold {fold_index}: v2 selected checkpoint provenance mismatch')
    adapter = folder / best['adapter_file']
    candidate_file = folder / best['candidates_file']
    if (file_sha(adapter) != best['adapter_sha256'] or
            file_sha(candidate_file) != best['candidates_sha256']):
        raise RuntimeError(f'Fold {fold_index}: v2 selected artifact changed')
    payload = read_json(candidate_file)
    if payload.get('config_hash') != V2_HASH or set(payload['candidates']) != set(FOLDS[fold_index]):
        raise RuntimeError(f'Fold {fold_index}: v2 candidate provenance changed')
    score = strict_macro(payload['candidates'], FOLDS[fold_index])['macro_f1']
    if abs(score - best['default_macro_f1']) > 1e-8:
        raise RuntimeError(f'Fold {fold_index}: v2 selected score did not reproduce')
    return adapter, payload['candidates'], best


V2_FOLDS = [selected_v2(i) for i in range(3)]
V2_OOF = {}
for _, candidates, _ in V2_FOLDS:
    V2_OOF.update(candidates)
if set(V2_OOF) != set(texts):
    raise RuntimeError('V2 fold candidates do not cover every labeled document')
v2_reproduced = strict_macro(V2_OOF, texts, V2_COMPARISON['v2_thresholds'])['macro_f1']
if abs(v2_reproduced - V2_COMPARISON['v2_calibrated_oof']) > 1e-8:
    raise RuntimeError('The saved v2 calibrated F1 did not reproduce')
print('V2 verified raw OOF:', f'{v2_reproduced:.6f}',
      'v2 reranked OOF:', f'{V2_BASELINE:.6f}')


class V3SpanExtractor(EnhancedSpanExtractor):
    def __init__(self, base):
        super().__init__(base)
        hidden = base.hidden_size
        self.biaffine_start = nn.Linear(hidden, 128, bias=False)
        self.biaffine_end = nn.Linear(hidden, 128, bias=False)
        self.biaffine_out = nn.Linear(128, len(LABELS))
        self.pair_residual = nn.Sequential(
            nn.Linear(5 * hidden + 16, CONFIG['v3_residual_hidden']),
            nn.GELU(), nn.Dropout(0.20),
            nn.Linear(CONFIG['v3_residual_hidden'], len(LABELS)))
        nn.init.zeros_(self.biaffine_out.weight)
        nn.init.zeros_(self.biaffine_out.bias)
        nn.init.zeros_(self.pair_residual[-1].weight)
        nn.init.zeros_(self.pair_residual[-1].bias)
        for block in base.blocks[-CONFIG['v3_unfreeze_top_blocks']:]:
            for parameter in block.parameters():
                parameter.requires_grad_(True)

    def score_pairs(self, hidden, pair_index):
        original = super().score_pairs(hidden, pair_index)
        width = hidden.shape[-1]
        def take(index):
            return hidden.gather(1, index.unsqueeze(-1).expand(-1, -1, width))
        start_index, end_index = pair_index[..., 0], pair_index[..., 1]
        start, end = take(start_index), take(end_index)
        before = take((start_index - 1).clamp_min(0))
        after = take((end_index + 1).clamp_max(hidden.shape[1]-1))
        distance = (end_index-start_index).clamp(min=0, max=CONFIG['max_span_tokens'])
        features = torch.cat((start, end, before, after, start*end,
                              self.width_embedding(distance)), dim=-1)
        interaction = self.biaffine_start(start) * self.biaffine_end(end)
        return original + self.biaffine_out(interaction) + self.pair_residual(features)


def v3_model(v2_adapter):
    record = torch.load(BASE_CHECKPOINT, map_location='cpu', weights_only=True)
    if record.get('model_source_sha256') != model_sha:
        raise RuntimeError('The Phase-1 base changed before v3 construction')
    base = model_module.build_model(record['model_config'])
    base.load_state_dict(record['model_state'], strict=True)
    del record
    model = V3SpanExtractor(base)
    saved = torch.load(v2_adapter, map_location='cpu', weights_only=True)
    if saved.get('config_hash') != V2_HASH:
        raise RuntimeError('The v3 warm adapter is not from the verified v2 run')
    current = model.state_dict()
    for name, value in saved['trainable_state'].items():
        if name not in current or current[name].shape != value.shape:
            raise RuntimeError(f'V2 warm state mismatch: {name}')
        current[name] = value
    model.load_state_dict(current, strict=True)
    print('V3 trainable parameters:',
          f'{sum(p.numel() for p in model.parameters() if p.requires_grad):,}',
          'of', f'{sum(p.numel() for p in model.parameters()):,}')
    return model


def verify_v3_best(folder, fold_index):
    path = folder / 'best.json'
    if not path.is_file():
        return None
    best = read_json(path)
    if best.get('config_hash') != CONFIG_HASH or best.get('fold') != fold_index:
        raise RuntimeError(f'Fold {fold_index}: v3 best provenance mismatch')
    for field, sha_field in [('adapter_file', 'adapter_sha256'),
                             ('candidates_file', 'candidates_sha256')]:
        if file_sha(folder / best[field]) != best[sha_field]:
            raise RuntimeError(f'Fold {fold_index}: v3 best SHA mismatch')
    candidates = read_json(folder / best['candidates_file'])
    if candidates.get('config_hash') != CONFIG_HASH or set(candidates['candidates']) != set(FOLDS[fold_index]):
        raise RuntimeError(f'Fold {fold_index}: v3 candidate IDs changed')
    score = strict_macro(candidates['candidates'], FOLDS[fold_index])['macro_f1']
    if abs(score - best['default_macro_f1']) > 1e-8:
        raise RuntimeError(f'Fold {fold_index}: v3 score did not reproduce')
    return best




## Resumable 12-layer continuation


In [ ]:
def v3_parameter_groups(model):
    groups = {'new': [], 'old_head': [], 'lora': [], 'new_encoder': [], 'old_encoder': []}
    cutoff = len(model.base.blocks) - CONFIG['unfreeze_top_blocks']
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        if name.startswith(('biaffine_start.', 'biaffine_end.', 'biaffine_out.',
                            'pair_residual.')):
            key = 'new'
        elif not name.startswith('base.'):
            key = 'old_head'
        elif '.a.' in name or '.b.' in name:
            key = 'lora'
        else:
            try:
                layer = int(name.split('.')[2]) if name.startswith('base.blocks.') else -1
            except (ValueError, IndexError):
                layer = -1
            key = 'old_encoder' if layer >= cutoff else 'new_encoder'
        groups[key].append(parameter)
    if any(not values for values in groups.values()):
        raise RuntimeError('One or more v3 trainable parameter groups are empty')
    rates = {'new': 1.2e-4, 'old_head': 4e-5, 'lora': 2e-5,
             'new_encoder': 8e-6, 'old_encoder': 4e-6}
    return [{'params': values, 'lr': rates[name], 'initial_lr': rates[name]}
            for name, values in groups.items()]


def exact_pair_rank_loss(output, batch):
    logits = output['pair'].float()
    positive = batch['pair_targets'].bool() & batch['pair_mask'].unsqueeze(-1)
    negative = (~batch['pair_targets'].bool()) & batch['pair_mask'].unsqueeze(-1)
    hardest = logits.masked_fill(~negative, -1e4).max(dim=1, keepdim=True).values
    values = F.softplus(0.50 + hardest - logits)
    return (values * positive).sum() / positive.sum().clamp_min(1)


def v3_loss(output, batch, class_weights):
    return task_loss(output, batch, class_weights) + 0.18 * exact_pair_rank_loss(output, batch)


def train_v3_fold(fold_index):
    folder = ROOT / f'fold{fold_index}'
    folder.mkdir(parents=True, exist_ok=True)
    summary_path, latest_path = folder/'summary.json', folder/'checkpoint_latest.pt'
    if summary_path.is_file():
        saved_summary = read_json(summary_path)
        if saved_summary.get('config_hash') != CONFIG_HASH:
            raise RuntimeError(f'Fold {fold_index}: incompatible resume summary')
        if saved_summary.get('status') == 'complete':
            verify_v3_best(folder, fold_index)
            print(f'Fold {fold_index}: verified complete')
            return saved_summary
        if saved_summary.get('status') == 'paused' and not latest_path.is_file():
            raise RuntimeError(f'Fold {fold_index}: paused without optimizer checkpoint')
    warm_adapter, old_candidates, old_best = V2_FOLDS[fold_index]
    heldout = FOLDS[fold_index]
    train_ids = set(texts) - set(heldout)
    rows = make_windows(train_ids, training=True)
    model = v3_model(warm_adapter).to('cuda:0')
    parallel = nn.DataParallel(model, device_ids=[0,1], output_device=0)
    optimizer = torch.optim.AdamW(v3_parameter_groups(model), weight_decay=0.01, foreach=False)
    total_steps = MAX_EPOCHS * math.ceil(len(rows)/CONFIG['batch_size'])
    warmup = max(1, int(total_steps*.04))
    def factor(step):
        if step < warmup:
            return max(.10, (step+1)/warmup)
        progress = (step-warmup)/max(1,total_steps-warmup)
        return max(.15, .5*(1+math.cos(math.pi*min(1,progress))))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, factor)
    weights = label_weights(train_ids).to('cuda:0')
    best = verify_v3_best(folder, fold_index)
    best_score = best['default_macro_f1'] if best else float(old_best['default_macro_f1'])
    history, epoch_start, stale = [],0,0
    warm_sha = file_sha(warm_adapter)
    if latest_path.is_file():
        saved = torch.load(latest_path, map_location='cpu', weights_only=False)
        if (saved.get('config_hash') != CONFIG_HASH or saved.get('fold') != fold_index or
                saved.get('warm_sha256') != warm_sha or saved.get('windows') != len(rows)):
            raise RuntimeError(f'Fold {fold_index}: optimizer provenance mismatch')
        load_trainable_state(model, saved['trainable_state'])
        optimizer.load_state_dict(saved['optimizer_state'])
        scheduler.load_state_dict(saved['scheduler_state'])
        torch.set_rng_state(saved['torch_rng_state'])
        torch.cuda.set_rng_state_all(saved['cuda_rng_states'])
        random.setstate(saved['python_rng_state'])
        history, epoch_start, stale = saved['history'], saved['epoch'], saved['stale']
        print(f'Fold {fold_index}: resuming after epoch {epoch_start}')
    paused=False
    try:
        for epoch in range(epoch_start+1, MAX_EPOCHS+1):
            if time.time()-RUN_STARTED > SESSION_GUARD_SECONDS:
                paused=True
                print('Time guard reached; Run All again in the same session to resume.')
                break
            order=list(range(len(rows)))
            random.Random(SEED + 300_000 + fold_index*1000 + epoch).shuffle(order)
            model.train()
            losses=[]
            for begin in range(0,len(order),CONFIG['batch_size']):
                group=[rows[i] for i in order[begin:begin+CONFIG['batch_size']]]
                batch={name:value.to('cuda:0',non_blocking=True)
                       for name,value in collate_windows(group,epoch+900).items()}
                optimizer.zero_grad(set_to_none=True)
                output=(parallel(batch['input_ids'],batch['attention_mask'],batch['pair_index'])
                        if len(group)>1 else
                        model(batch['input_ids'],batch['attention_mask'],batch['pair_index']))
                loss=v3_loss(output,batch,weights)
                if not torch.isfinite(loss):
                    raise FloatingPointError(f'Fold {fold_index} epoch {epoch}: nonfinite loss')
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                optimizer.step();scheduler.step()
                losses.append(float(loss.detach().cpu()))
            entry={'epoch':epoch,'train_loss_mean':sum(losses)/len(losses),
                   'elapsed_seconds':round(time.time()-RUN_STARTED,1)}
            if epoch%VALIDATE_EVERY==0 or epoch==MAX_EPOCHS:
                candidates=predict_documents(model,parallel,heldout,texts)
                score_report=strict_macro(candidates,heldout)
                score=score_report['macro_f1']
                entry['heldout_default_macro_f1']=score
                entry['candidate_recall_macro']=sum(
                    row['exact_candidate_recall'] for row in score_report['per_label'].values())/len(LABELS)
                if score>best_score+0.001:
                    adapter_name=f'selected_epoch{epoch:02d}.pt'
                    candidates_name=f'candidates_epoch{epoch:02d}.json'
                    atomic_torch(folder/adapter_name,{'config_hash':CONFIG_HASH,'fold':fold_index,
                                                      'epoch':epoch,'trainable_state':trainable_state(model),
                                                      'labels':LABELS})
                    atomic_json(folder/candidates_name,{'config_hash':CONFIG_HASH,'fold':fold_index,
                                                        'epoch':epoch,'candidates':candidates})
                    proposed={'config_hash':CONFIG_HASH,'fold':fold_index,'epoch':epoch,
                              'default_macro_f1':score,'adapter_file':adapter_name,
                              'adapter_sha256':file_sha(folder/adapter_name),
                              'candidates_file':candidates_name,
                              'candidates_sha256':file_sha(folder/candidates_name)}
                    previous=best
                    atomic_json(folder/'best.json',proposed)
                    best=verify_v3_best(folder,fold_index)
                    if previous is not None:
                        (folder/previous['adapter_file']).unlink(missing_ok=True)
                        (folder/previous['candidates_file']).unlink(missing_ok=True)
                    best_score=score;stale=0
                else:
                    stale+=1
                print(f'Fold {fold_index} epoch {epoch}: loss {entry["train_loss_mean"]:.4f}, '
                      f'F1 {score:.4f}, best {best_score:.4f}, '
                      f'free {free_gib():.2f} GiB',flush=True)
            else:
                print(f'Fold {fold_index} epoch {epoch}: loss {entry["train_loss_mean"]:.4f}',flush=True)
            history.append(entry)
            atomic_json(folder/'history.json',history)
            if free_gib()<MIN_FREE_GIB:
                raise RuntimeError('Disk reserve reached before v3 optimizer checkpoint')
            atomic_torch(latest_path,{'config_hash':CONFIG_HASH,'fold':fold_index,
                                      'warm_sha256':warm_sha,'windows':len(rows),
                                      'epoch':epoch,'stale':stale,'history':history,
                                      'trainable_state':trainable_state(model),
                                      'optimizer_state':optimizer.state_dict(),
                                      'scheduler_state':scheduler.state_dict(),
                                      'torch_rng_state':torch.get_rng_state(),
                                      'cuda_rng_states':torch.cuda.get_rng_state_all(),
                                      'python_rng_state':random.getstate()})
            if epoch>=MIN_EPOCHS and stale>=PATIENCE:
                print(f'Fold {fold_index}: early stopping after {stale} non-improving validations')
                break
        if paused and not latest_path.is_file():
            raise RuntimeError('Time guard reached before a resumable checkpoint was made')
        best=verify_v3_best(folder,fold_index)
        result={'status':'paused' if paused else 'complete','config_hash':CONFIG_HASH,
                'fold':fold_index,'warm_default_macro_f1':old_best['default_macro_f1'],
                'selected_source':'v3' if best else 'v2',
                'selected_epoch':best['epoch'] if best else 0,
                'selected_default_macro_f1':best_score,'history':history}
        atomic_json(summary_path,result)
        if not paused:latest_path.unlink(missing_ok=True)
        return result
    finally:
        del model,parallel,optimizer,scheduler
        torch.cuda.empty_cache()


FOLD_RESULTS={}
for fold_index in range(3):
    if time.time()-RUN_STARTED>SESSION_GUARD_SECONDS:
        print('Time guard before another fold; Run All again to resume.')
        break
    FOLD_RESULTS[fold_index]=train_v3_fold(fold_index)
    if FOLD_RESULTS[fold_index]['status']!='complete':break




## Cross-validated source selection


In [ ]:
def selected_v3_candidates(fold_index):
    best=verify_v3_best(ROOT/f'fold{fold_index}',fold_index)
    if best is None:return V2_FOLDS[fold_index][1]
    return read_json(ROOT/f'fold{fold_index}'/best['candidates_file'])['candidates']


def load_v1_candidates():
    config=read_json(V1_ROOT/'experiment_config.json')
    digest=hashlib.sha256(json.dumps(config,sort_keys=True).encode()).hexdigest()
    if (config.get('base_checkpoint_sha256')!=base_sha or
            config.get('data_hash')!=data_hash):
        raise RuntimeError('V1 candidate source differs from the official data or base')
    output={}
    for fold_index in range(3):
        folder=V1_ROOT/f'fold{fold_index}'
        best=read_json(folder/'best.json')
        if best.get('config_hash')!=digest or best.get('fold')!=fold_index:
            raise RuntimeError(f'Fold {fold_index}: V1 candidate provenance mismatch')
        path=folder/best['candidates_file']
        if file_sha(path)!=best['candidates_sha256']:
            raise RuntimeError(f'Fold {fold_index}: V1 candidate SHA mismatch')
        group=read_json(path)['candidates']
        if set(group)!=set(FOLDS[fold_index]):
            raise RuntimeError(f'Fold {fold_index}: V1 candidate IDs mismatch')
        output.update(group)
    return output


def crossfit_reranker(lower_map, upper_map):
    X,y,rows,ranges=build_feature_table(texts,lower_map,upper_map,texts,with_gold=True)
    print('Cross-fit candidate feature table:',X.shape,'positive exact:',int(y.sum()),flush=True)
    output={}
    for fold_index in range(3):
        validation=FOLDS[fold_index]
        training=sorted(set(texts)-set(validation))
        model=train_reranker(X,y,ranges,training)
        output.update(rerank_documents(model,X,rows,ranges,validation))
        del model
        print('Cross-fit reranker fold',fold_index,'done',flush=True)
    if set(output)!=set(texts):
        raise RuntimeError('Cross-fitted reranker is missing labeled documents')
    del X,y,rows,ranges
    return output


def label_source_choices(sources, thresholds):
    reports={key:strict_macro(candidate_map,texts,thresholds[key])
             for key,candidate_map in sources.items()}
    choice={}
    for label in LABELS:
        baseline=reports['v2_reranked']['per_label'][label]['f1']
        candidate=max(sources,key=lambda key:reports[key]['per_label'][label]['f1'])
        advantage=reports[candidate]['per_label'][label]['f1']-baseline
        choice[label]=candidate if advantage>=.015 else 'v2_reranked'
    return choice,reports


def normalized_for_threshold(score,threshold):
    s=min(1-1e-6,max(1e-6,float(score)))
    t=min(1-1e-6,max(1e-6,float(threshold)))
    margin=math.log(s/(1-s))-math.log(t/(1-t))
    return 1/(1+math.exp(-max(-40,min(40,margin))))


def hybrid_candidates(sources,source_thresholds,choices,doc_ids):
    output={}
    for doc_id in doc_ids:
        selected=[]
        for source_name,candidate_map in sources.items():
            thresholds=source_thresholds[source_name]
            for a,b,label,score in candidate_map[doc_id]:
                if choices[label]!=source_name:continue
                confidence=normalized_for_threshold(score,thresholds[label])
                if confidence>=.5:
                    selected.append([int(a),int(b),label,float(confidence)])
        selected.sort(key=lambda row:-row[3])
        output[doc_id]=selected
    return output


SELECTION=None
V3_OOF=None
V2_RERANK_OOF=None
V3_RERANK_OOF=None
if len(FOLD_RESULTS)==3 and all(x['status']=='complete' for x in FOLD_RESULTS.values()):
    V3_OOF={}
    for fold_index in range(3):V3_OOF.update(selected_v3_candidates(fold_index))
    if set(V3_OOF)!=set(texts):
        raise RuntimeError('V3 OOF candidate map is incomplete')
    v3_default=strict_macro(V3_OOF,texts)['macro_f1']
    v3_thresholds,v3_calibrated=calibrate(V3_OOF)
    print('V3 selected-fold F1: default',f'{v3_default:.6f}',
          'calibrated',f'{v3_calibrated:.6f}',flush=True)

    V1_OOF=load_v1_candidates()
    V2_RERANK_OOF=crossfit_reranker(V1_OOF,V2_OOF)
    v2_rr_thresholds=V2_COMPARISON['reranker_thresholds']
    reproduced=fast_macro(V2_RERANK_OOF,v2_rr_thresholds)
    if abs(reproduced-V2_BASELINE)>1e-8:
        raise RuntimeError(f'The saved v2 cross-fit reranker score did not reproduce: {reproduced}')
    del V1_OOF

    V3_RERANK_OOF=crossfit_reranker(V2_OOF,V3_OOF)
    v3_rr_default=fast_macro(V3_RERANK_OOF,{})
    v3_rr_thresholds,v3_rr_calibrated=calibrate(V3_RERANK_OOF)
    sources={'v2_raw':V2_OOF,'v2_reranked':V2_RERANK_OOF,
             'v3_raw':V3_OOF,'v3_reranked':V3_RERANK_OOF}
    source_thresholds={
        'v2_raw':V2_COMPARISON['v2_thresholds'],
        'v2_reranked':v2_rr_thresholds,
        'v3_raw':v3_thresholds,'v3_reranked':v3_rr_thresholds}
    choices,reports=label_source_choices(sources,source_thresholds)
    hybrid=hybrid_candidates(sources,source_thresholds,choices,texts)
    common_thresholds={label:.5 for label in LABELS}
    hybrid_report=strict_macro(hybrid,texts,common_thresholds)
    hybrid_score=hybrid_report['macro_f1']
    alternatives={'v2_reranked':V2_BASELINE,'v3_raw':v3_calibrated,
                  'v3_reranked':v3_rr_calibrated,'hybrid':hybrid_score}
    method=max(alternatives,key=alternatives.get)
    fold_checks=[]
    for fold_index in range(3):
        ids=FOLDS[fold_index]
        fold_checks.append({
            'fold':fold_index,
            'v2_reranked':fast_macro({d:V2_RERANK_OOF[d] for d in ids},v2_rr_thresholds),
            'v3_raw':fast_macro({d:V3_OOF[d] for d in ids},v3_thresholds),
            'v3_reranked':fast_macro({d:V3_RERANK_OOF[d] for d in ids},v3_rr_thresholds),
            'hybrid':fast_macro({d:hybrid[d] for d in ids},common_thresholds)})
    SELECTION={
        'status':'complete','config_hash':CONFIG_HASH,
        'baseline_v2_reranked_oof':V2_BASELINE,
        'v2_raw_calibrated_oof':V2_COMPARISON['v2_calibrated_oof'],
        'v3_default_oof':v3_default,'v3_calibrated_oof':v3_calibrated,
        'v3_reranker_default_oof':v3_rr_default,
        'v3_reranker_calibrated_oof':v3_rr_calibrated,
        'hybrid_oof':hybrid_score,'chosen_method':method,
        'chosen_oof':alternatives[method],
        'v3_thresholds':v3_thresholds,
        'v3_reranker_thresholds':v3_rr_thresholds,
        'hybrid_label_sources':choices,
        'hybrid_source_thresholds':source_thresholds,
        'fold_checks':fold_checks,
        'per_label':{key:report['per_label'] for key,report in reports.items()},
        'per_label_hybrid':hybrid_report['per_label'],
        'caveat':'Fold checkpoint, per-label source, and threshold selection reuse OOF labels; local F1 is optimistic.'}
    atomic_json(ROOT/'model_selection.json',SELECTION)
    print('OOF baseline',f'{V2_BASELINE:.6f}','v3',f'{v3_calibrated:.6f}',
          'v3 reranked',f'{v3_rr_calibrated:.6f}',
          'hybrid',f'{hybrid_score:.6f}','chosen',method,flush=True)
    del hybrid,reports,choices,sources,V2_RERANK_OOF,V3_RERANK_OOF
else:
    print('The three v3 folds are incomplete. Run All in this session to resume.')




## Final model and test candidates


In [ ]:
def selected_final_epochs():
    epochs=sorted(int(FOLD_RESULTS[i]['selected_epoch']) for i in range(3)
                  if FOLD_RESULTS[i]['selected_source']=='v3')
    return max(2,min(MAX_EPOCHS,epochs[len(epochs)//2])) if epochs else 0


def train_v3_final(epochs):
    final_dir=ROOT/'final'
    final_dir.mkdir(parents=True,exist_ok=True)
    summary_path=final_dir/'summary.json'
    latest_path=final_dir/'checkpoint_latest.pt'
    adapter_path=final_dir/'selected_adapter.pt'
    warm_path=V2_ROOT/'final'/'selected_adapter.pt'
    warm_sha=file_sha(warm_path)
    if warm_sha!=V2_FINAL['adapter_sha256']:
        raise RuntimeError('V2 full-data warm adapter changed')
    if summary_path.is_file():
        previous=read_json(summary_path)
        if previous.get('config_hash')!=CONFIG_HASH or previous.get('epochs')!=epochs:
            raise RuntimeError('Incompatible v3 final-model resume summary')
        if previous['status']=='complete':
            if file_sha(adapter_path)!=previous['adapter_sha256']:
                raise RuntimeError('The completed v3 final adapter changed')
            print('V3 final fit verified complete')
            return previous
    rows=make_windows(texts,training=True)
    model=v3_model(warm_path).to('cuda:0')
    parallel=nn.DataParallel(model,device_ids=[0,1],output_device=0)
    optimizer=torch.optim.AdamW(v3_parameter_groups(model),weight_decay=.01,foreach=False)
    total_steps=max(1,epochs*math.ceil(len(rows)/CONFIG['batch_size']))
    scheduler=torch.optim.lr_scheduler.LambdaLR(
        optimizer,lambda step:max(.15,.5*(1+math.cos(math.pi*min(1,step/total_steps)))))
    weights=label_weights(texts).to('cuda:0')
    history=[];epoch_start=0
    if latest_path.is_file():
        saved=torch.load(latest_path,map_location='cpu',weights_only=False)
        if (saved.get('config_hash')!=CONFIG_HASH or saved.get('epochs')!=epochs or
                saved.get('warm_sha256')!=warm_sha or saved.get('windows')!=len(rows)):
            raise RuntimeError('Final v3 optimizer checkpoint provenance mismatch')
        load_trainable_state(model,saved['trainable_state'])
        optimizer.load_state_dict(saved['optimizer_state'])
        scheduler.load_state_dict(saved['scheduler_state'])
        torch.set_rng_state(saved['torch_rng_state'])
        torch.cuda.set_rng_state_all(saved['cuda_rng_states'])
        random.setstate(saved['python_rng_state'])
        history,epoch_start=saved['history'],saved['epoch']
        print('Resuming v3 final fit after epoch',epoch_start)
    paused=False
    try:
        for epoch in range(epoch_start+1,epochs+1):
            if time.time()-RUN_STARTED>SESSION_GUARD_SECONDS:
                paused=True
                print('Time guard during final fit; Run All again to resume.')
                break
            order=list(range(len(rows)))
            random.Random(SEED+900_000+epoch).shuffle(order)
            model.train();losses=[]
            for begin in range(0,len(order),CONFIG['batch_size']):
                group=[rows[i] for i in order[begin:begin+CONFIG['batch_size']]]
                batch={name:value.to('cuda:0',non_blocking=True)
                       for name,value in collate_windows(group,epoch+1200).items()}
                optimizer.zero_grad(set_to_none=True)
                output=(parallel(batch['input_ids'],batch['attention_mask'],batch['pair_index'])
                        if len(group)>1 else
                        model(batch['input_ids'],batch['attention_mask'],batch['pair_index']))
                loss=v3_loss(output,batch,weights)
                if not torch.isfinite(loss):
                    raise FloatingPointError(f'Final epoch {epoch}: nonfinite loss')
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                optimizer.step();scheduler.step()
                losses.append(float(loss.detach().cpu()))
            history.append({'epoch':epoch,'train_loss_mean':sum(losses)/len(losses),
                            'elapsed_seconds':round(time.time()-RUN_STARTED,1)})
            atomic_json(final_dir/'history.json',history)
            if free_gib()<MIN_FREE_GIB:
                raise RuntimeError('Disk reserve reached before final optimizer save')
            atomic_torch(latest_path,{'config_hash':CONFIG_HASH,'epochs':epochs,
                                      'warm_sha256':warm_sha,'windows':len(rows),
                                      'epoch':epoch,'history':history,
                                      'trainable_state':trainable_state(model),
                                      'optimizer_state':optimizer.state_dict(),
                                      'scheduler_state':scheduler.state_dict(),
                                      'torch_rng_state':torch.get_rng_state(),
                                      'cuda_rng_states':torch.cuda.get_rng_state_all(),
                                      'python_rng_state':random.getstate()})
            print(f'Final v3 epoch {epoch}/{epochs}: loss {history[-1]["train_loss_mean"]:.4f}, '
                  f'free {free_gib():.2f} GiB',flush=True)
        if not paused:
            atomic_torch(adapter_path,{'config_hash':CONFIG_HASH,'epochs':epochs,
                                       'trainable_state':trainable_state(model),'labels':LABELS})
        result={'status':'paused' if paused else 'complete','config_hash':CONFIG_HASH,
                'epochs':epochs,'history':history,
                'adapter_sha256':file_sha(adapter_path) if not paused else None}
        atomic_json(summary_path,result)
        if not paused:latest_path.unlink(missing_ok=True)
        return result
    finally:
        del model,parallel,optimizer,scheduler
        torch.cuda.empty_cache()


def infer_v3_test(final_summary):
    final_dir=ROOT/'final'
    partial=final_dir/'test_candidates.partial.json'
    provenance_path=final_dir/'test_candidates_provenance.json'
    provenance={'config_hash':CONFIG_HASH,
                'adapter_sha256':final_summary['adapter_sha256'],
                'test_sha256':file_sha(SOURCE_DIR/'fold3-text.json')}
    if provenance_path.is_file() and read_json(provenance_path)!=provenance:
        raise RuntimeError('V3 test cache provenance changed')
    atomic_json(provenance_path,provenance)
    existing=read_json(partial) if partial.is_file() else {}
    if not set(existing)<=set(test_texts):
        raise RuntimeError('V3 partial test inference has unknown document IDs')
    warm_path=V2_ROOT/'final'/'selected_adapter.pt'
    model=v3_model(warm_path).to('cuda:0')
    saved=torch.load(final_dir/'selected_adapter.pt',map_location='cpu',weights_only=True)
    if saved.get('config_hash')!=CONFIG_HASH:
        raise RuntimeError('V3 selected adapter configuration changed')
    load_trainable_state(model,saved['trainable_state'])
    parallel=nn.DataParallel(model,device_ids=[0,1],output_device=0)
    try:
        def save_progress(predictions):
            if len(predictions)%5==0:
                atomic_json(partial,predictions)
                print('V3 test inference',len(predictions),'/',len(test_texts),flush=True)
        raw=predict_documents(model,parallel,test_texts,test_texts,
                              existing=existing,on_document=save_progress)
    finally:
        del model,parallel
        torch.cuda.empty_cache()
    if set(raw)!=set(test_texts):
        raise RuntimeError('V3 inference did not finish all test documents')
    atomic_json(partial,raw)
    return raw


FINAL_RESULT=None
V3_TEST_RAW=None
FINAL_GATE=max(MIN_FINAL_CV_F1,V2_BASELINE+.002)
if SELECTION is not None:
    if SELECTION['chosen_oof']<FINAL_GATE:
        print(f'Selected local F1 {SELECTION["chosen_oof"]:.4f} did not clear '
              f'{FINAL_GATE:.4f}; keep the 0.5069 submission.')
    elif time.time()-RUN_STARTED>SESSION_GUARD_SECONDS:
        print('Time guard before full-data fit; Run All again to continue.')
    else:
        uses_v3=(SELECTION['chosen_method'] in ('v3_raw','v3_reranked') or
                 (SELECTION['chosen_method']=='hybrid' and
                  any(source.startswith('v3_') for source in
                      SELECTION['hybrid_label_sources'].values())))
        epochs=selected_final_epochs() if uses_v3 else 0
        if uses_v3 and epochs:
            FINAL_RESULT=train_v3_final(epochs)
            if FINAL_RESULT['status']=='complete':
                V3_TEST_RAW=infer_v3_test(FINAL_RESULT)
        else:
            FINAL_RESULT={'status':'complete','epochs':0,
                          'note':'Selected decoder uses only verified v2 candidate states.'}
            V3_TEST_RAW=read_json(V2_TEST_PATH)
        if V3_TEST_RAW is not None and set(V3_TEST_RAW)!=set(test_texts):
            raise RuntimeError('Final v3 test candidate IDs are incomplete')




## Verified package and report


In [ ]:
PACKAGE=None
if V3_TEST_RAW is not None and SELECTION is not None:
    v1_test_path=V1_ROOT/'final'/'test_candidates.partial.json'
    v2_test=read_json(V2_TEST_PATH)
    v1_test=read_json(v1_test_path)
    if set(v1_test)!=set(test_texts) or set(v2_test)!=set(test_texts):
        raise RuntimeError('The older test candidate caches are incomplete')

    # Reproduce the exact existing 0.5069 submission before combining sources.
    baseline_meta_path=V2_ROOT/'final'/'document_reranker.pkl'
    if not baseline_meta_path.is_file():
        raise RuntimeError('The saved v2 document reranker is missing')
    with baseline_meta_path.open('rb') as stream:
        baseline_meta=pickle.load(stream)
    X_test,_,baseline_rows,baseline_ranges=build_feature_table(
        test_texts,v1_test,v2_test,test_texts,with_gold=False)
    v2_reranked_test=rerank_documents(
        baseline_meta,X_test,baseline_rows,baseline_ranges,test_texts)
    baseline_predictions=decode(v2_reranked_test,V2_COMPARISON['reranker_thresholds'])
    if baseline_predictions!=read_json(V2_ROOT/'final'/'prediction.json'):
        raise RuntimeError('The existing v2 submission did not reproduce; refusing to mix candidates')
    del baseline_meta,X_test,baseline_rows,baseline_ranges
    print('Existing 0.5069 prediction.json reproduced exactly')

    # The second reranker is trained on all public OOF candidates only.
    X_train,y_train,_,train_ranges=build_feature_table(
        texts,V2_OOF,V3_OOF,texts,with_gold=True)
    new_meta=train_reranker(X_train,y_train,train_ranges,sorted(texts))
    meta_path=ROOT/'final'/'document_reranker.pkl'
    meta_path.parent.mkdir(parents=True,exist_ok=True)
    with tempfile.NamedTemporaryFile(dir=meta_path.parent,suffix='.tmp',delete=False) as stream:
        temporary=Path(stream.name)
    try:
        with temporary.open('wb') as stream:
            pickle.dump(new_meta,stream,protocol=pickle.HIGHEST_PROTOCOL)
        os.replace(temporary,meta_path)
    finally:
        temporary.unlink(missing_ok=True)
    del X_train,y_train,train_ranges
    X_test,_,new_rows,new_ranges=build_feature_table(
        test_texts,v2_test,V3_TEST_RAW,test_texts,with_gold=False)
    v3_reranked_test=rerank_documents(new_meta,X_test,new_rows,new_ranges,test_texts)
    del new_meta,X_test,new_rows,new_ranges

    method=SELECTION['chosen_method']
    source_maps={'v2_raw':v2_test,'v2_reranked':v2_reranked_test,
                 'v3_raw':V3_TEST_RAW,'v3_reranked':v3_reranked_test}
    if method=='v3_raw':
        predictions=decode(V3_TEST_RAW,SELECTION['v3_thresholds'])
    elif method=='v3_reranked':
        predictions=decode(v3_reranked_test,SELECTION['v3_reranker_thresholds'])
    elif method=='hybrid':
        combined=hybrid_candidates(
            source_maps,SELECTION['hybrid_source_thresholds'],
            SELECTION['hybrid_label_sources'],test_texts)
        predictions=decode(combined,{label:.5 for label in LABELS})
    else:
        raise RuntimeError('An unchanged v2 method cannot create a new submission')
    if set(predictions)!=set(test_texts):
        raise RuntimeError('The new submission lacks an official test document')
    for doc_id,spans in predictions.items():
        for a,b,label in spans:
            if label not in LABEL_ID or not 0<=a<b<=len(test_texts[doc_id]):
                raise RuntimeError(f'Invalid v3 span in test document {doc_id}')
        if any(spans[i][1]>spans[i+1][0] for i in range(len(spans)-1)):
            raise RuntimeError(f'Overlapping v3 predictions in test document {doc_id}')
    output=ROOT/'final'/'prediction.json'
    atomic_json(output,predictions)
    submission=WORKING/'CS781_Phase2_Span_V3_Submission.zip'
    temporary=submission.with_suffix('.zip.tmp')
    with zipfile.ZipFile(temporary,'w',compression=zipfile.ZIP_DEFLATED,
                         compresslevel=6) as archive:
        archive.write(output,arcname='prediction.json')
    os.replace(temporary,submission)
    with zipfile.ZipFile(submission) as archive:
        if (archive.testzip() is not None or archive.namelist()!=['prediction.json'] or
                json.loads(archive.read('prediction.json'))!=predictions):
            raise RuntimeError('V3 submission ZIP validation failed')
    PACKAGE={'status':'READY_FOR_MANUAL_UPLOAD','method':method,
             'submission_zip':str(submission),'zip_sha256':file_sha(submission),
             'test_documents':len(predictions),
             'predicted_spans':sum(map(len,predictions.values()))}
    atomic_json(ROOT/'final'/'package.json',PACKAGE)
    print('New verified submission:',submission)

RECOVERY=None
members=[(ROOT/'experiment_config.json','experiment_config.json')]
for fold_index in range(3):
    best=verify_v3_best(ROOT/f'fold{fold_index}',fold_index)
    if best is not None:
        members.append((ROOT/f'fold{fold_index}'/best['adapter_file'],
                        f'fold{fold_index}/selected_adapter.pt'))
for name in ('selected_adapter.pt','document_reranker.pkl'):
    path=ROOT/'final'/name
    if path.is_file():members.append((path,f'final/{name}'))
if len(members)>1:
    recovery=WORKING/'CS781_Phase2_Span_V3_Recovery.zip'
    temporary=recovery.with_suffix('.zip.tmp')
    with zipfile.ZipFile(temporary,'w',compression=zipfile.ZIP_DEFLATED,
                         compresslevel=1) as archive:
        archive.writestr('README.txt',
                         'V3 trainable states only. The Phase-1 base-03 and v2 adapters are '
                         'required for reconstruction. Use the V3 notebook source.\n')
        for path,name in members:archive.write(path,arcname=name)
    os.replace(temporary,recovery)
    with zipfile.ZipFile(recovery) as archive:
        if archive.testzip() is not None:
            raise RuntimeError('V3 recovery ZIP CRC failure')
    RECOVERY={'zip':str(recovery),'sha256':file_sha(recovery),
              'members':[name for _,name in members]}

status=('READY_FOR_MANUAL_UPLOAD' if PACKAGE else
        'PAUSED_RUN_ALL_TO_RESUME' if SELECTION is None or
        (SELECTION['chosen_oof']>=FINAL_GATE and V3_TEST_RAW is None)
        else 'NO_LOCAL_IMPROVEMENT_KEEP_V2')
REPORT={'version':VERSION,'status':status,'config_hash':CONFIG_HASH,
        'elapsed_seconds':round(time.time()-RUN_STARTED,1),
        'disk_free_gib':round(free_gib(),3),
        'v2_official_score_user_reported':0.506919,
        'v2_baseline_oof':V2_BASELINE,'final_fit_gate':FINAL_GATE,
        'fold_results':FOLD_RESULTS,'selection':SELECTION,
        'final_fit':FINAL_RESULT,'package':PACKAGE,'recovery':RECOVERY,
        'caveats':[
            'The official private F1 for this new submission is unknown.',
            'Best epochs, thresholds, and hybrid label choices were selected using the same OOF labels.',
            'Cross-fitted reranker training uses public labeled documents only.',
            'The 0.506919 baseline was reported by the user; prediction.json was reproduced from saved artifacts.',
            'No outside pretrained weights or private annotations were used.']}
atomic_json(ROOT/'report.json',REPORT)
report_files=[(ROOT/'report.json','report.json'),
              (ROOT/'experiment_config.json','experiment_config.json')]
if (ROOT/'model_selection.json').is_file():
    report_files.append((ROOT/'model_selection.json','model_selection.json'))
for fold_index in range(3):
    folder=ROOT/f'fold{fold_index}'
    for name in ('summary.json','history.json','best.json'):
        path=folder/name
        if path.is_file():report_files.append((path,f'fold{fold_index}/{name}'))
    best=verify_v3_best(folder,fold_index)
    if best is not None:
        report_files.append((folder/best['candidates_file'],
                             f'fold{fold_index}/{best["candidates_file"]}'))
for name in ('summary.json','package.json','test_candidates.partial.json'):
    path=ROOT/'final'/name
    if path.is_file():report_files.append((path,f'final/{name}'))
temporary=REPORT_ZIP.with_suffix('.zip.tmp')
with zipfile.ZipFile(temporary,'w',compression=zipfile.ZIP_DEFLATED,
                     compresslevel=6) as archive:
    archive.writestr('README.txt',
                     'Inspect report.json and model_selection.json. '
                     'Submit only when status is READY_FOR_MANUAL_UPLOAD. '
                     'The prior v2 score remains available.\n')
    for path,name in report_files:archive.write(path,arcname=name)
os.replace(temporary,REPORT_ZIP)
with zipfile.ZipFile(REPORT_ZIP) as archive:
    if archive.testzip() is not None:
        raise RuntimeError('V3 report ZIP CRC failure')
print('Report:',REPORT_ZIP,'status:',status)
if PACKAGE:print('Submission:',PACKAGE['submission_zip'])
if RECOVERY:print('Recovery:',RECOVERY['zip'])
print('Keep the old 0.5069 submission until a new official score is known.')
